- https://www.kaggle.com/code/juyeon0815/exercise-introduction 바탕으로 작성
- 모델 시나리오 : 아이오와 주의 주택 가격을 예측

---

### [1단계] 데이터 불러오기 및 전처리

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 읽기
X_full = pd.read_csv('data/train.csv')
X_test_full = pd.read_csv('data/test.csv')

# 타켓 변수 및 특징 선택
y = X_full.SalePrice
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = X_full[features].copy()
X_test = X_test_full[features].copy()

# 훈련 데이터와 검증 데이터로 분할
X_train, X_valid, y_train, y_valid = train_test_split(
    X,y, test_size=0.2, train_size=0.8, random_state=0
)

### [2단계] 랜덤 포레스트 모델 정의

In [8]:
from sklearn.ensemble import RandomForestRegressor

model_1 = RandomForestRegressor(n_estimators=50, random_state=0)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)
model_3 = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
model_4 = RandomForestRegressor(n_estimators=200, min_samples_split=20, random_state=0)
model_5 = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)

models = [model_1, model_2, model_3, model_4, model_5]

### [3단계] : 모델 평가 (MAE계산)

In [9]:
from sklearn.metrics import mean_absolute_error

# 모델 평가 함수 정의
def score_model(model, X_t=X_train, X_v=X_valid, y_t=y_train, y_v=y_valid):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

# 모델별 MAE 출력
for i in range(len(models)):
    mae = score_model(models[i])
    print(f"Model {i+1} MAE: {mae}")


Model 1 MAE: 24015.492818003917
Model 2 MAE: 23740.979228636657
Model 3 MAE: 23528.78421232877
Model 4 MAE: 23996.676789668687
Model 5 MAE: 23706.672864217904


### [4단계] : 테스트 데이터에 대한 예측 생성
랜덤 포레스트 모델을 사용하여 테스트 데이터를 예측합니다.

In [11]:
# 모델 정의
my_model = RandomForestRegressor()

# 모델 학습
my_model.fit(X, y)

# 테스트 데이터 예측
preds_test = my_model.predict(X_test)

# 예측 결과 저장 (CSV 파일 생성)
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': preds_test})
output.to_csv('data/submission.csv', index=False)
